## Hybrid Search in Azure Cosmos DB

### Install Libraries and Utilities

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up the Environment

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Creating the Cosmos DB Client

In [ ]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Creating the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Creating the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Creating the Full-Text Search Index

In [ ]:
full_text_container_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [
        {
            "path": "/content",
            "language": "en-US"
        }
    ]
}

full_text_indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/_etag/?"
        },
        {
            "path": "/vector/*"
        }
    ],
    "fullTextIndexes": [
        {
            "path": "/content",
            "language": "en-US"
        }
    ],
    "vectorIndexes": [
        {
            "path": "/vector",
            "type": "diskANN"
        }
    ]
}

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path":"/vector",
            "dataType":"float32",
            "distanceFunction":"cosine",
            "dimensions":1536
        }
    ]
}

container_properties = container.read()

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = full_text_indexing_policy,
    full_text_policy = full_text_container_policy,
    vector_embedding_policy = vector_embedding_policy
)

print("Full-text search index created successfully!")

### Run a Simple Full-Text Search Query

In [ ]:
full_text_query = """SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.content
FROM c
ORDER BY RANK FullTextScore(
    c.content,
    @query
)"""

results = container.query_items(
    query=full_text_query,
    parameters=[
        {
            "name": "@query",
            "value": "protein rich smoothies with raw cacao"
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}, content: {}".format(item['name'], item['category'], item['content']))
    print("-----------")
    print("content: {}".format(item['content']))
    print("====================================")
    print("\n\n")

### Run a Vector Search Query

In [ ]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.content,
    VectorDistance(
        c.vector,
        @queryVector
    ) AS SimilarityScore
FROM c
ORDER BY VectorDistance(
    c.vector,
    @queryVector
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "protein rich smoothies with raw cacao")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}, similarity score: {}".format(item['name'], item['category'], item['SimilarityScore']))
    print("-----------")
    print("content: {}".format(item['content']))
    print("====================================")
    print("\n\n")

### Run a Hybrid Search Query with RRF (Reciprocal Rank Fusion)

In [ ]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.content
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.vector, @queryVector),
    FullTextScore(c.content, @queryText)
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "protein rich smoothies with raw cacao")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        },
        {
            "name": "@queryText",
            "value": "protein rich smoothies with raw cacao"
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}".format(item['name'], item['category']))
    print("-----------")
    print("content: {}".format(item['content']))
    print("====================================")
    print("\n\n")

### Run a Weighted Hybrid Search Query

In [ ]:
query = """
SELECT TOP 5
    c.id,
    c.name,
    c.category,
    c.content
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.vector, @queryVector),
    FullTextScore(c.content, @queryText),
    [1,2]
)
"""

# try some other query text too like "chocolate dessert"
query_embedding = generate_embeddings(azure_openai_client, "protein rich smoothies with raw cacao")

results = container.query_items(
    query=query,
    parameters=[
        {
            "name": "@queryVector",
            "value": query_embedding
        },
        {
            "name": "@queryText",
            "value": "protein rich smoothies with raw cacao"
        }
    ],
    enable_cross_partition_query=True
)

for item in results:
    print("name: {}, category: {}".format(item['name'], item['category']))
    print("-----------")
    print("content: {}".format(item['content']))
    print("====================================")
    print("\n\n")